# Milestone 3 lab, notebook 1 of 8: A price has a start and an end

## Where we are

| Notebook | What you do |
|---|---|
| **1. A price has a start and an end** (you are here) | products. See one forgotten date count every sale twice |
| 2. Six rows, wrong everywhere | stores. Find out why the weakest check is the best one here |
| 3. A gap is not a zero | stock_counts. Look for a row that is not there |
| 4. The same person twice | members. Find one person behind two keys, and keep their data private |
| 5. A file you do not control | supplier_prices. A supplier renames a column, and nothing fails |
| 6. Same shape, new meaning | supplier_prices. Every number is valid, and 100 times too high |
| 7. A contract for the supplier | Write down what a file must look like, and check it at the door |
| 8. Cards side by side | Compare the seven cards. Then see what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import (
    keep, lab_copy, money, on_lab, plant, prepare_week, production, rebuild_sales_lines,
    rebuilt_against_real, show_card, sql, standard_checks,
)

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 8 clean tables.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: Where you stopped

**Lotus Retails** is a chain of six supermarkets. Customers pay at a **till**: the machine at the counter where a customer pays. It is also called a cash register.

In milestone 2 you wrote sixteen checks for `sales_lines` and its control, `till_totals`. They find every fault planted in the sales files.

Lotus has five more tables. This milestone gives each one its own check card, and from the card its own checks. Start with `products`: the price list.

Here is what is at stake. The buying team types this table by hand. In this notebook you will forget **one date** in one row. Every carton of eggs sold in three weeks, in all six stores, will be counted twice. That is $3,259.32 of sales that never happened, and six managers paid a bonus on it.

Not one of your sixteen checks looks at `products`.

## Step 2: The card first. What does one row mean?

You know the method from milestone 1: fill in the check card before you write a check. Question 1 is the grain.

Look at three products.

In [2]:
sql("""
    SELECT product_id, product_name, unit_price_cents, valid_from, valid_to, status
    FROM clean.products
    WHERE product_id IN ('P-003', 'P-006', 'P-011')
    ORDER BY product_id, valid_from
""")

,product_id,product_name,unit_price_cents,valid_from,valid_to,status
0,P-003,Butter 250g,349,2026-01-05,2026-08-27,active
1,P-003,Butter 250g,399,2026-08-28,NaN,active
2,P-006,"Eggs, 12",489,2026-01-05,2026-08-09,active
3,P-006,"Eggs, 12",519,2026-08-10,NaN,active
4,P-011,"Bagels, 6",359,2026-01-05,2026-08-16,retired


Eggs are there twice, and that is right. The price went from $4.89 to $5.19 on 10 August. The old row ends on 9 August. The new row starts on 10 August and has no end yet.

`valid_from` and `valid_to` are the row's **validity dates**: the first and the last day the row is true. An empty `valid_to` means "still true today".

Butter changed on 28 August. Bagels were retired: their last day was 16 August.

So one row is **not** one product. One row is one product at one price.

In [3]:
show_card("products", 1)

**From the check card `checks/products.md`, question 1. What does one row mean?**

One product at one price. A product whose price changed once has two rows: the old price with its last day, and the new price with its first day.

The key is `product_id` + `valid_from`. `product_id` alone is **not** the key.

Now try the check everybody writes first on a table like this: "the product id is unique".

In [4]:
sql("""
    SELECT product_id, count(*) AS times
    FROM clean.products
    GROUP BY product_id
    HAVING count(*) > 1
""")

,product_id,times
0,P-006,2
1,P-003,2


It fails on two rows, and both are right. That check has the wrong grain. On this table it would fail every night from the first price change on, and people would switch it off.

The opposite of a row with validity dates is a row that is simply true, like a sales line. A sale happened. It has no start and no end.

## Step 3: What would hurt most?

Questions 3 and 4 of the card ask how the table is made, and which fault hurts most. A person types it. So the faults are a person's faults.

In [5]:
show_card("products", 4)

**From the check card `checks/products.md`, question 4. Which of those would hurt most?**

| Rank | Worry | Why it is here |
|---|---|---|
| 1 | Two prices on one day | Every sale of that product is counted twice, in every store, and no row looks wrong |
| 2 | A day with no price | Every sale of that product is dropped, in every store |
| 3 | A price typed wrong | Every till charges it. Customers notice before the data team does |
| 4 | A product that is not typed yet | Its sales are dropped. This is planted case 6, and check 5 on `sales_lines` catches it |
| 5 | A retired product with no last day | It stays on the order list |
| 6 | An untidy product name | It hurts nobody. **No check** |

## Step 4: Plant the first fault. A row that was not closed

When a price changes, a person must do two things: close the old row, and open a new one. Suppose they open the new Eggs row and forget to close the old one.

You do this in a private copy. `lab_copy()` makes `lab.products`, and `plant()` changes only the copy.

In [6]:
lab_copy("clean.products")
plant("""
    UPDATE lab.products SET valid_to = NULL
    WHERE product_id = 'P-006' AND valid_from = DATE '2026-01-05'
""")

lab.products is a fresh copy of clean.products: 42 rows.
Planted in the lab copy.


In [7]:
sql("SELECT product_id, unit_price_cents, valid_from, valid_to FROM lab.products WHERE product_id = 'P-006' ORDER BY valid_from")

,product_id,unit_price_cents,valid_from,valid_to
0,P-006,489,2026-01-05,None
1,P-006,519,2026-08-10,None


Two rows, both open. From 10 August on, Eggs have **two prices on the same day**.

Every row still looks fine alone: a real product, a real price, a real date. The fault is between two rows.

The pipeline joins every sales line to the price that is valid on its day. Run that exact join again, with your copy of the price list.

In [8]:
rebuild_sales_lines()

The real clean.sales_lines has 24,384 rows.
Rebuilt with your lab copy: 24,765 rows. That is +381.


381 rows more. Every Eggs line from 10 August found two prices, so the join made two rows of it.

What does that do to the money? Here are the five weeks of each store, from the real table and from the rebuilt one.

In [9]:
compare = rebuilt_against_real()
compare

,store_id,real,rebuilt,difference
0,S01,"$33,427.35","$34,070.91",$643.56
1,S02,"$30,143.22","$30,786.78",$643.56
2,S03,"$26,855.52","$27,462.75",$607.23
3,S04,"$26,229.55","$26,629.18",$399.63
4,S05,"$21,784.12","$22,251.22",$467.10
5,S06,"$23,537.17","$24,035.41",$498.24


In [10]:
too_much = sql("""
    SELECT (SELECT sum(amount_cents) FROM lab.sales_lines)
         - (SELECT sum(amount_cents) FROM clean.sales_lines) AS cents
""")
print("Counted too much, in all six stores:", money(too_much["cents"][0]))

Counted too much, in all six stores: $3,259.32


All six stores are too high. One forgotten date, and $3,259.32 of eggs is counted twice.

Would the pipeline notice? Run its five standard checks on the rebuilt table.

In [11]:
standard_checks(on_the_lab=True)

1. The table is not empty           0 bad rows  PASS


2. The key is not null              0 bad rows  PASS
3. The key is unique                381 bad rows  FAIL
4. The status is an allowed value   0 bad rows  PASS
5. The total is positive            0 bad rows  PASS


Check 3 fails. Good: the night would stop.

But read what it says. 381 bad rows, in `sales_lines`. Somebody now opens the sales files of six stores and looks for lines sent twice. There are none. The sales files are perfect. The fault is one row in a table nobody is looking at.

A check that fails far from the cause costs hours. So check the price list **itself**.

## Step 5: Write the check. Row against neighbouring row

Take every price row of a product, and look at its later rows. Does a later row start while this one is still valid?

That is a join of the table to itself.

In [12]:
ONE_PRICE_AT_A_TIME = """
    SELECT a.product_id AS thing, b.valid_from AS day, a.product_name,
           a.valid_from AS first_from, a.valid_to AS first_to, b.valid_from AS second_from
    FROM clean.products a
    JOIN clean.products b
      ON b.product_id = a.product_id AND b.valid_from > a.valid_from
     AND b.valid_from <= coalesce(a.valid_to, DATE '9999-12-31')
"""
sql(on_lab(ONE_PRICE_AT_A_TIME))

,thing,day,product_name,first_from,first_to,second_from
0,P-006,2026-08-10,"Eggs, 12",2026-01-05,None,2026-08-10


One row, and it names the product and the day the two prices start to overlap. That is where to look.

On the real table it must be silent.

In [13]:
keep("one_price_at_a_time", sql(ONE_PRICE_AT_A_TIME))

Your query:           0 bad rows
The production check: 0 bad rows  (A product has one price on any day)
The same rows.


## Step 6: Plant the opposite fault. A day with no price

Now the other half of the same job. The old row is closed on 9 August, as it should be. But the new row is typed two days late: it starts on 12 August.

On 10 and 11 August, Eggs have no price at all.

In [14]:
lab_copy("clean.products")
plant("""
    UPDATE lab.products SET valid_from = DATE '2026-08-12'
    WHERE product_id = 'P-006' AND valid_to IS NULL
""")
rebuild_sales_lines()

lab.products is a fresh copy of clean.products: 42 rows.
Planted in the lab copy.


The real clean.sales_lines has 24,384 rows.
Rebuilt with your lab copy: 24,348 rows. That is -36.


In [15]:
rebuilt_against_real()

,store_id,real,rebuilt,difference
0,S01,"$33,427.35","$33,328.74",-$98.61
1,S02,"$30,143.22","$30,065.37",-$77.85
2,S03,"$26,855.52","$26,803.62",-$51.90
3,S04,"$26,229.55","$26,198.41",-$31.14
4,S05,"$21,784.12","$21,747.79",-$36.33
5,S06,"$23,537.17","$23,490.46",-$46.71


36 rows fewer. The join found no price for those lines, so it dropped them. $342.54 of real sales is gone, spread over all six stores.

And this time?

In [16]:
standard_checks(on_the_lab=True)

1. The table is not empty           0 bad rows  PASS


2. The key is not null              0 bad rows  PASS


3. The key is unique                0 bad rows  PASS
4. The status is an allowed value   0 bad rows  PASS
5. The total is positive            0 bad rows  PASS


**Five green checks.** A dropped row cannot fail a check: it is not there.

The check for it reads the price list the same way, row against the next row. For every price that ends: does another start the very next day?

In [17]:
NO_GAP_BETWEEN_PRICES = """
    SELECT a.product_id AS thing, a.valid_to + 1 AS day, a.product_name, a.valid_to AS price_ends
    FROM clean.products a
    LEFT JOIN clean.products b
      ON b.product_id = a.product_id AND b.valid_from = a.valid_to + 1
    WHERE a.valid_to IS NOT NULL AND a.status = 'active' AND b.product_id IS NULL
"""
sql(on_lab(NO_GAP_BETWEEN_PRICES))

,thing,day,product_name,price_ends
0,P-006,2026-08-10,"Eggs, 12",2026-08-09


It returns Eggs, and the first day with no price: 10 August.

Notice `a.status = 'active'`. Bagels end on 16 August and nothing follows. That is right: they were retired. Without that word the check would fire on good data.

In [18]:
keep("no_gap_between_prices", sql(NO_GAP_BETWEEN_PRICES))

Your query:           0 bad rows
The production check: 0 bad rows  (A product on sale has a price on every day)
The same rows.


## Step 7: One more. A slip of the finger

The third worry on the card: a price typed wrong. Someone means 519 and types 5190. Every till in the company now charges $51.90 for eggs.

In [19]:
lab_copy("clean.products")
plant("UPDATE lab.products SET unit_price_cents = 5190 WHERE product_id = 'P-006' AND valid_to IS NULL")
print(len(production("product_values_allowed", on_the_lab=True)), "bad rows from the check 'a product row makes sense'")

lab.products is a fresh copy of clean.products: 42 rows.


Planted in the lab copy.
0 bad rows from the check 'a product row makes sense'


The "makes sense" check is silent. 5190 is a perfectly allowed price: it is a number, and it is above zero. Salmon costs $11.95.

Compare the new price with the price before it.

In [20]:
PRICE_CHANGE_IS_BELIEVABLE = """
    SELECT new.product_id AS thing, new.valid_from AS day, new.product_name,
           old.unit_price_cents AS old_price_cents, new.unit_price_cents AS new_price_cents
    FROM clean.products new
    JOIN clean.products old
      ON old.product_id = new.product_id AND old.valid_to + 1 = new.valid_from
    WHERE new.unit_price_cents > 2 * old.unit_price_cents
       OR new.unit_price_cents * 2 < old.unit_price_cents
"""
sql(on_lab(PRICE_CHANGE_IS_BELIEVABLE))

,thing,day,product_name,old_price_cents,new_price_cents
0,P-006,2026-08-10,"Eggs, 12",489,5190


In [21]:
keep("price_change_is_believable", sql(PRICE_CHANGE_IS_BELIEVABLE))

Your query:           0 bad rows
The production check: 0 bad rows  (A new price is near the old price)
The same rows.


The limit is "at most twice, at least half". A real price can double, so this check is a reason for a person to look, not a proof.

## Step 8: What must be prepared

Look at what all three checks did first.

In [22]:
show_card("products", 5)

**From the check card `checks/products.md`, question 5. What must be prepared before it can be checked?**

| Worry | What must be prepared first |
|---|---|
| Two prices on one day | Join the table to itself: each price row against the later rows of the same product |
| A day with no price | For each price that ends, look for the one that starts the next day |
| A price typed wrong | Join each price to the price before it |

A table of prices is checked against **itself**, row against neighbouring row. Nothing like that was needed for `sales_lines`.

The checks for `sales_lines` compared the table with a till, with a price list, with a normal day. A price list has no till and no normal day. It is checked against **itself**: each row against the row next to it.

The card has five checks in all. The other two are in `lotus_pipeline/other_checks.py`: a product row makes sense, and a retired product has a last day.

## What you now know

- The grain of `products` is one product at one price. The key is `product_id` + `valid_from`.
- **Validity dates** are the first and the last day a row is true.
- Two prices on one day count every sale twice. A day with no price drops every sale, and all five standard checks stay green.
- A table with validity dates is checked row against neighbouring row: no overlap, no gap, no wild jump.

**Say it in an interview:** "Our price list had validity dates, so its key was the product and the start date. I checked it against itself: no two prices of a product on one day, and no day without a price. In a test, one old row left open counted $3,259 of egg sales twice in six stores, and the only check that failed pointed at the sales table, where nothing was wrong."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), sections "What is good data?" and "A starter kit for any table".

Next: [notebook 2](02_six_rows_wrong_everywhere.ipynb). A table with six rows. What could there be to check?